In [0]:
%pip install geopandas fiona shapely pyproj --quiet
dbutils.library.restartPython()

In [0]:
import os
import zipfile
import urllib.request
import tempfile

# Natural Earth urban areas datasets
datasets = {
    "ne_10m_urban_areas": "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_urban_areas.zip",
    "ne_50m_urban_areas": "https://naciscdn.org/naturalearth/50m/cultural/ne_50m_urban_areas.zip",
}

download_dir = tempfile.mkdtemp(prefix="natural_earth_")

for name, url in datasets.items():
    zip_path = os.path.join(download_dir, f"{name}.zip")
    extract_dir = os.path.join(download_dir, name)
    
    print(f"Downloading {name}...")
    urllib.request.urlretrieve(url, zip_path)
    
    print(f"Extracting {name}...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)
    
    print(f"  -> Extracted to {extract_dir}")
    print(f"  -> Files: {os.listdir(extract_dir)}")

print(f"\nAll downloads complete. Base dir: {download_dir}")

In [0]:
import geopandas as gpd

# Read the 10m urban areas shapefile (highest detail)
shp_path_10m = os.path.join(download_dir, "ne_10m_urban_areas", "ne_10m_urban_areas.shp")
gdf_10m = gpd.read_file(shp_path_10m)

print(f"10m Urban Areas: {len(gdf_10m)} features")
print(f"CRS: {gdf_10m.crs}")
print(f"\nColumns: {list(gdf_10m.columns)}")
print(f"\nGeometry types: {gdf_10m.geometry.geom_type.value_counts().to_dict()}")
gdf_10m.head()

In [0]:
# Read the 50m urban areas shapefile
shp_path_50m = os.path.join(download_dir, "ne_50m_urban_areas", "ne_50m_urban_areas.shp")
gdf_50m = gpd.read_file(shp_path_50m)

print(f"50m Urban Areas: {len(gdf_50m)} features")
print(f"CRS: {gdf_50m.crs}")
print(f"\nColumns: {list(gdf_50m.columns)}")
gdf_50m.head()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
from shapely import wkt
import pandas as pd

def geodataframe_to_spark(gdf, dataset_name):
    """
    Convert a GeoDataFrame to a Spark DataFrame.
    Geometry is stored as WKT string for compatibility.
    """
    # Convert geometry to WKT
    pdf = gdf.copy()
    pdf["geometry_wkt"] = pdf.geometry.apply(lambda g: g.wkt if g is not None else None)
    pdf = pdf.drop(columns=["geometry"])
    
    # Convert any remaining non-serializable types
    for col in pdf.columns:
        if pdf[col].dtype == "object":
            pdf[col] = pdf[col].astype(str)
    
    spark_df = spark.createDataFrame(pdf)
    print(f"{dataset_name}: converted {spark_df.count()} rows, {len(spark_df.columns)} columns")
    return spark_df

# Convert both datasets
spark_10m = geodataframe_to_spark(gdf_10m, "ne_10m_urban_areas")
spark_50m = geodataframe_to_spark(gdf_50m, "ne_50m_urban_areas")

display(spark_10m.limit(5))

### Configure your target catalog and schema below
Update the `catalog` and `schema` variables to match your Unity Catalog destination.

In [0]:
# ---- UPDATE THESE TO YOUR TARGET LOCATION ----
catalog = "cmegdemos_catalog"
schema = "geospatial_analytics"

# ------------------------------------------------

# spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

# Write 10m urban areas
table_10m = f"{catalog}.{schema}.ne_10m_urban_areas"
spark_10m.write.mode("overwrite").saveAsTable(table_10m)
print(f"Wrote {spark.table(table_10m).count()} rows to {table_10m}")

# Write 50m urban areas
table_50m = f"{catalog}.{schema}.ne_50m_urban_areas"
spark_50m.write.mode("overwrite").saveAsTable(table_50m)
print(f"Wrote {spark.table(table_50m).count()} rows to {table_50m}")

print("\nIngestion complete!")

In [0]:
# Quick verification
print("=== 10m Urban Areas ===")
display(spark.sql(f"SELECT * FROM {table_10m} LIMIT 5"))

print(f"\n=== 50m Urban Areas ===")
display(spark.sql(f"SELECT * FROM {table_50m} LIMIT 5"))

In [0]:
# Cleanup temp files
import shutil
shutil.rmtree(download_dir, ignore_errors=True)
print(f"Cleaned up temp directory: {download_dir}")

In [0]:
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm
from shapely import wkt
import tempfile, os, zipfile, urllib.request

# Read 10m urban areas from Delta
pdf_10m = spark.table("cmegdemos_catalog.geospatial_analytics.ne_10m_urban_areas").toPandas()
pdf_10m["geometry"] = pdf_10m["geometry_wkt"].apply(wkt.loads)
gdf_10m = gpd.GeoDataFrame(pdf_10m, geometry="geometry", crs="EPSG:4326")

# Download 50m states/provinces for US state borders
tmp = tempfile.mkdtemp()
states_url = "https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_1_states_provinces.zip"
states_zip = os.path.join(tmp, "states.zip")
urllib.request.urlretrieve(states_url, states_zip)
with zipfile.ZipFile(states_zip, "r") as z:
    z.extractall(tmp)
states = gpd.read_file(os.path.join(tmp, "ne_50m_admin_1_states_provinces.shp"))
us_states = states[states["admin"] == "United States of America"]

# Filter urban areas to CONUS bounding box
us_bbox = (-130, 24, -65, 50)
gdf_us = gdf_10m.cx[us_bbox[0]:us_bbox[2], us_bbox[1]:us_bbox[3]]

# Plot
fig, ax = plt.subplots(1, 1, figsize=(20, 12))

# State boundaries
us_states.plot(ax=ax, color="#f7f7f7", edgecolor="#999999", linewidth=0.5)

# Urban areas colored by area_sqkm
norm = Normalize(vmin=gdf_us["area_sqkm"].quantile(0.05), vmax=gdf_us["area_sqkm"].quantile(0.95))
gdf_us.plot(ax=ax, column="area_sqkm", cmap="YlOrRd", norm=norm, alpha=0.8, edgecolor="#c1272d", linewidth=0.2)

# Colorbar
sm = cm.ScalarMappable(cmap="YlOrRd", norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.02, pad=0.02)
cbar.set_label("Urban Area (sq km)", fontsize=11)

ax.set_xlim(-130, -65)
ax.set_ylim(24, 50)
ax.set_title("United States Urban Areas — Natural Earth 10m Detail", fontsize=16, fontweight="bold", pad=15)
ax.set_axis_off()

plt.tight_layout()
plt.show()

# Cleanup
import shutil
shutil.rmtree(tmp, ignore_errors=True)

print(f"{len(gdf_us)} urban area polygons in CONUS")
print(f"Total urban area: {gdf_us['area_sqkm'].sum():,.0f} sq km")
print(f"Largest: {gdf_us['area_sqkm'].max():,.0f} sq km | Smallest: {gdf_us['area_sqkm'].min():,.1f} sq km")

In [0]:
# Zoom into Washington State
fig, ax = plt.subplots(1, 1, figsize=(16, 10))

# State boundaries
us_states.plot(ax=ax, color="#f7f7f7", edgecolor="#999999", linewidth=0.8)

# Washington State bounding box
wa_bbox = (-125, 45.5, -116.9, 49.1)
gdf_wa = gdf_10m.cx[wa_bbox[0]:wa_bbox[2], wa_bbox[1]:wa_bbox[3]]

# Urban areas colored by size
norm = Normalize(
    vmin=gdf_wa["area_sqkm"].quantile(0.05),
    vmax=gdf_wa["area_sqkm"].quantile(0.95),
)
gdf_wa.plot(ax=ax, column="area_sqkm", cmap="YlOrRd", norm=norm, alpha=0.8, edgecolor="#c1272d", linewidth=0.4)

# Colorbar
sm = cm.ScalarMappable(cmap="YlOrRd", norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.02, pad=0.02)
cbar.set_label("Urban Area (sq km)", fontsize=11)

ax.set_xlim(wa_bbox[0], wa_bbox[2])
ax.set_ylim(wa_bbox[1], wa_bbox[3])
ax.set_title("Washington State \u2014 Urban Areas (10m Detail)", fontsize=16, fontweight="bold", pad=15)
ax.set_axis_off()

plt.tight_layout()
plt.show()

print(f"{len(gdf_wa)} urban area polygons in Washington State")
print(f"Total urban area: {gdf_wa['area_sqkm'].sum():,.0f} sq km")

---
## FCC 5G NR Coverage Data — Washington State
Broadband Data Collection (BDC) mobile broadband coverage at **H3 resolution 9** hexagons.

Source: `/Volumes/cmegdemos_catalog/geospatial_analytics/raw_data/bdc_53_5GNR_mobile_broadband_h3_J25_14apr2026.zip`

| Field | Description |
|---|---|
| `technology` | FCC tech code (500 = 5G NR) |
| `mindown` | Minimum advertised download speed (Mbps) |
| `minup` | Minimum advertised upload speed (Mbps) |
| `environmnt` | 0 = Outdoor, 1 = Indoor |
| `h3_res9_id` | H3 resolution 9 hex cell ID |

In [0]:
import zipfile, tempfile, os
import geopandas as gpd

# Extract GeoPackage from zip
zip_path = "/Volumes/cmegdemos_catalog/geospatial_analytics/raw_data/bdc_53_5GNR_mobile_broadband_h3_J25_14apr2026.zip"
extract_dir = tempfile.mkdtemp(prefix="bdc_5g_")
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

gpkg_path = os.path.join(extract_dir, "bdc_53_5GNR_mobile_broadband_h3_J25_14apr2026.gpkg")
gdf_5g = gpd.read_file(gpkg_path)

print(f"Total H3 cells: {len(gdf_5g):,}")
print(f"CRS: {gdf_5g.crs}")
print(f"\nEnvironment breakdown:")
print(f"  Indoor (1):  {(gdf_5g['environmnt'] == 1).sum():,} cells")
print(f"  Outdoor (0): {(gdf_5g['environmnt'] == 0).sum():,} cells")
print(f"\nDownload speeds (Mbps): min={gdf_5g['mindown'].min()}, median={gdf_5g['mindown'].median()}, max={gdf_5g['mindown'].max()}")
print(f"Upload speeds (Mbps):   min={gdf_5g['minup'].min()}, median={gdf_5g['minup'].median()}, max={gdf_5g['minup'].max()}")

display(gdf_5g.head(10))

In [0]:
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, ListedColormap
from matplotlib import cm
import numpy as np

# Use outdoor coverage for the map (environment=0)
gdf_outdoor = gdf_5g[gdf_5g["environmnt"] == 0].copy()

fig, ax = plt.subplots(1, 1, figsize=(18, 12))

# State boundaries (reuse from earlier cells if available, otherwise download)
try:
    us_states.plot(ax=ax, color="#f7f7f7", edgecolor="#999999", linewidth=0.8)
except NameError:
    import urllib.request
    tmp2 = tempfile.mkdtemp()
    states_url = "https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_1_states_provinces.zip"
    states_zip = os.path.join(tmp2, "states.zip")
    urllib.request.urlretrieve(states_url, states_zip)
    with zipfile.ZipFile(states_zip, "r") as z:
        z.extractall(tmp2)
    states = gpd.read_file(os.path.join(tmp2, "ne_50m_admin_1_states_provinces.shp"))
    us_states = states[states["admin"] == "United States of America"]
    us_states.plot(ax=ax, color="#f7f7f7", edgecolor="#999999", linewidth=0.8)
    import shutil
    shutil.rmtree(tmp2, ignore_errors=True)

# Plot 5G outdoor coverage colored by download speed
norm = Normalize(vmin=7, vmax=35)
gdf_outdoor.plot(
    ax=ax, column="mindown", cmap="plasma", norm=norm,
    alpha=0.6, edgecolor="none", linewidth=0
)

# Overlay urban areas from earlier if available
try:
    gdf_wa = gdf_10m.cx[-125:-116.9, 45.5:49.1]
    gdf_wa.boundary.plot(ax=ax, edgecolor="#2ecc71", linewidth=0.8, alpha=0.7, label="Urban Areas")
except NameError:
    from shapely import wkt
    pdf_10m = spark.table("cmegdemos_catalog.geospatial_analytics.ne_10m_urban_areas").toPandas()
    pdf_10m["geometry"] = pdf_10m["geometry_wkt"].apply(wkt.loads)
    gdf_10m = gpd.GeoDataFrame(pdf_10m, geometry="geometry", crs="EPSG:4326")
    gdf_wa = gdf_10m.cx[-125:-116.9, 45.5:49.1]
    gdf_wa.boundary.plot(ax=ax, edgecolor="#2ecc71", linewidth=0.8, alpha=0.7, label="Urban Areas")

# Colorbar
sm = cm.ScalarMappable(cmap="plasma", norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.02, pad=0.02)
cbar.set_label("Min Download Speed (Mbps)", fontsize=11)

# Legend for urban area outlines
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color="#2ecc71", linewidth=1.5, label="Urban Area Boundaries")]
ax.legend(handles=legend_elements, loc="lower left", fontsize=11)

ax.set_xlim(-125, -116.9)
ax.set_ylim(45.5, 49.1)
ax.set_title("5G NR Outdoor Coverage — Washington State\nFCC BDC H3 res9 · Min Download Speed (Mbps)", fontsize=15, fontweight="bold", pad=15)
ax.set_axis_off()

plt.tight_layout()
plt.show()

print(f"{len(gdf_outdoor):,} outdoor H3 cells plotted")
print(f"Coverage area: ~{len(gdf_outdoor) * 0.1053:.0f} sq km (H3 res9 ≈ 0.1053 km² each)")

next steps:
- create zoomable map

- use catalog cmegdemos_catalog and geospatial_analytics schema (parameterize these values throughout the notebook)
- Focusing on the Seattle market, do the following:
- Loading FCC BDC H3 hexagon coverage data and OpenCelliD tower points as spatial tables - save to delta tables in UC (zip_path = "/Volumes/cmegdemos_catalog/geospatial_analytics/raw_data/bdc_53_5GNR_mobile_broadband_h3_J25_14apr2026.zip")
- Importing Microsoft building footprints as polygon geometries using Databricks Spatial SQL and save to delta table (zip path: /Volumes/cmegdemos_catalog/geospatial_analytics/raw_data/Washington.zip)
- Performing spatial joins to determine which buildings fall within specific coverage hexagons or calculating distance to nearest towers
- Analyzing signal strength predictions based on tower proximity, building density, and terrain using geospatial functions
